## Convolution from Scratch

In [1]:
import numpy as np

In [2]:
#Define a simple 5x5 grayscale image
#since grayscale hai to 3sra koi dimension hi nahi hoga
image=np.array([
    [1,2,3,0,1],
    [0,1,2,3,2],
    [3,0,1,2,1],
    [2,1,3,0,0],
    [1,2,0,1,2]
])

#Define a 3x3 kernel(basically filter)
kernel=np.array([
    [0,1,0],
    [1,-4,1],
    [0,1,0]
])

#Define convolution operation
def convolution(image,kernel):
  image_h,image_w=image.shape
  kernel_h,kernel_w=kernel.shape
  #we know the math behind it like the output of the matrix would be (n,m)-(k1,k2)+1
  #more of like assume i as a sliding windows qki isme humlog wahi krte hia
  output_h=image_h-kernel_h+1
  output_w=image_w-kernel_w+1
  output=np.zeros((output_h,output_w))
  for i in range(output_h):
    for j in range(output_w):
      output[i,j]=np.sum(image[i:i+kernel_h,j:j+kernel_w]*kernel)
  return output

answer=convolution(image,kernel)
print(answer)

[[  0.   0.  -6.]
 [  6.   3.  -3.]
 [  3. -10.   6.]]


# Padding with stride=1

1. Padding: Zero-padding is added to the image to control the spatial dimensions of the output.

2. Stride: Controls how much the filter moves at eah step. A stride of 1 means the filter moves one pixel at a time.

3. Convolution with Padding and Stride: The output is now controlled by the padding and stride,allowing us to maintain or reduce the spatial dimensions of the output


In [3]:
print(image)

[[1 2 3 0 1]
 [0 1 2 3 2]
 [3 0 1 2 1]
 [2 1 3 0 0]
 [1 2 0 1 2]]


In [4]:
import numpy as np

#Adding padding

#here pad is given as pad_width like i have readed it on documentation that this is repsonsible as
#Number of values padded to the edges of each axis unique pad widths for each axis
#and here we passed mode='contant' that means that pads with a constant value ,
#there are other ways too like :
#'edge':Pads with the edge values of the array
#'linear_ramp':Pads with the linear ramp between end_value and the array edge value
#'maximum': Pads with the maximum value of all or part of the vector along each axis
#'mean':Pads with the mean of all or part of the vector along each axis
#'median':Pads with the median of all or part of the vector along each axis
#'minimum':Pads with the minimum value of all or part of the vector along each axis
#'reflect':Pads with the reflection of the array
#'symmetric':Pads with the reflection of the array
#'wrap':Pads with the wrap of the array
def pad_image(image,pad):
  return np.pad(image,pad,mode='constant',constant_values=0) #This is like 0 padding

#Modify convolution to include stride and padding
def convolve_with_padding_and_stride(image,kernel,stride=1,padding=0):
  original_image_h,original_image_w=image.shape # Store original dimensions
  kernel_h,kernel_w=kernel.shape

  if padding>0:
    padded_image = pad_image(image,padding)
  else:
    padded_image = image

  # Calculate output dimensions based on padded image dimensions
  # (Input_Dimension - Kernel_Dimension + 2 * Padding) / Stride + 1
  output_h=(padded_image.shape[0]-kernel_h)//stride+1
  output_w=(padded_image.shape[1]-kernel_w)//stride+1

  output=np.zeros((output_h,output_w))

  for i in range(0,output_h*stride,stride):
    for j in range(0,output_w*stride,stride):
      output[i//stride,j//stride]=np.sum(padded_image[i:i+kernel_h,j:j+kernel_w]*kernel)
  return output


#Apply the Convolution with padding and stride
padded_output=convolve_with_padding_and_stride(image,kernel,stride=1,padding=1)
print(padded_output)


[[ -2.  -3.  -8.   7.  -2.]
 [  5.   0.   0.  -6.  -3.]
 [-10.   6.   3.  -3.   0.]
 [ -3.   3. -10.   6.   3.]
 [  0.  -6.   6.  -2.  -7.]]


In [5]:
print(pad_image(image,1))

[[0 0 0 0 0 0 0]
 [0 1 2 3 0 1 0]
 [0 0 1 2 3 2 0]
 [0 3 0 1 2 1 0]
 [0 2 1 3 0 0 0]
 [0 1 2 0 1 2 0]
 [0 0 0 0 0 0 0]]


## Max Pooling

In [6]:
#Max Pooling Function
def max_pooling(image,pool_size,stride):
  image_h,image_w=image.shape
  output_h=(image_h-pool_size)//stride + 1
  output_w=(image_w-pool_size)//stride + 1
  output=np.zeros((output_h,output_w))

  for i in range(0,output_h*stride,stride):
    for j in range(0,output_w*stride,stride):
      output[i//stride,j//stride]=np.max(image[i:i+pool_size,j:j+pool_size])
  return output


# Apply max pooling to the original 'image' variable
pooled_output=max_pooling(image,2,2)
print(pooled_output)


[[2. 3.]
 [3. 3.]]


## Convolution on RGB

In [7]:

# Define a simple 5x5x3 RGB image (3 channels)
image = np.array([
    [[1, 0, 2], [2, 1, 1], [3, 2, 0], [0, 1, 1], [1, 0, 2]],
    [[0, 1, 0], [1, 0, 1], [2, 2, 2], [3, 1, 3], [2, 0, 1]],
    [[3, 0, 2], [0, 1, 0], [1, 0, 1], [2, 2, 2], [1, 0, 0]],
    [[2, 1, 1], [1, 0, 2], [3, 3, 1], [0, 1, 0], [0, 2, 1]],
    [[1, 2, 2], [2, 1, 0], [0, 0, 1], [1, 2, 2], [2, 1, 1]]
])

# Define a 3x3x3 filter (kernel) for each channel (RGB)
kernel = np.array([
    [[0, 1, 0], [1, -1, 1], [0, 1, 0]],
    [[1, 0, 1], [0, -1, 0], [1, 0, 1]],
    [[0, 1, 0], [1, 1, 1], [0, 1, 0]]
])

# Convolution operation
def convolve_rgb(image, kernel):
    image_h, image_w, image_c = image.shape
    kernel_h, kernel_w, kernel_c = kernel.shape
    output_h = image_h - kernel_h + 1
    output_w = image_w - kernel_w + 1
    output = np.zeros((output_h, output_w, image_c))

    for k in range(image_c): # Apply the convolution for each channel
        for i in range(output_h):
            for j in range(output_w):
                output[i, j, k] = np.sum(image[i:i+kernel_h, j:j+kernel_w, k] * kernel[:, :, k])

    return output

# Run the function
result = convolve_rgb(image, kernel)
print(result)

[[[4. 2. 3.]
  [8. 1. 5.]
  [6. 2. 6.]]

 [[6. 6. 6.]
  [7. 3. 5.]
  [5. 5. 4.]]

 [[7. 2. 2.]
  [2. 3. 4.]
  [6. 0. 6.]]]


## Implementing the same with keras


In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models


#Example input shape for a 32x32 RGB Image
input_shape=(100,100,3)

#Define the model
model=models.Sequential()

#Add Input layer
model.add(layers.Input(shape=input_shape))

#Add Convolutional layer with
model.add(layers.Conv2D(filters=16,kernel_size=(3,3),activation='relu'))

#Add Max Pooling layer
model.add(layers.MaxPooling2D(pool_size=(2,2)))

#Adding more convolution and pooling layera as needed
#Yaha pe filters mtlb number of kernels
model.add(layers.Conv2D(filters=32,kernel_size=(3,3),activation='relu'))
model.add(layers.MaxPooling2D(pool_size=(2,2)))

#Print the Model Summary to see the structure
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 98, 98, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 49, 49, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 47, 47, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 23, 23, 32)     │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,088 (19.88 KB)

 Trainable params: 5,088 (19.88 KB)

 Non-trainable params: 0 (0.00 B)